In [ ]:
import pysammos.data_read.mfix.file_read as fr
from pysammos.data_read.mfix.point_data import get_point_data_variable

In [78]:
import numpy as np
from vtk.util.numpy_support import vtk_to_numpy
import vtk

In [91]:
def reader(file_type: str, path: str) -> vtk.vtkAlgorithmOutput:
    """
    Reads the VTK file using the appropriate reader based on the detected file type.

    Parameters
    ----------
    file_type : str
        The type of VTK file: "vtp" for PolyData or "vtk" for UnstructuredGrid.
    path : str
        The path to the VTK file.

    Returns
    -------
    vtk.vtkAlgorithmOutput
        The output data from the VTK reader.

    Raises
    ------
    ValueError
        If the file format is unsupported.
    FileNotFoundError
        If the specified file does not exist.
    IOError
        If the file cannot be opened or read.
    
    Examples
    --------
    >>> file_type = get_file_type("example.vtp")
    XML-based PolyData detected.
    >>> reader_output = reader(file_type, "example.vtp")
    >>> print(type(reader_output))
    <class 'vtkmodules.vtkCommonExecutionModel.vtkAlgorithmOutput'>
    This indicates that the file has been read successfully and the output is a VTK algorithm output object.
    
    """

    # Use the appropriate reader
    if file_type == "vtp":
        reader = vtk.vtkXMLPolyDataReader()
    elif file_type == "vtk":
        reader = vtk.vtkUnstructuredGridReader()
    else:
        raise ValueError(f"(reader) Unsupported file format, '{file_type}'. Supported formats: 'vtp', 'vtk'")

    # Check if the file exists and is accessible
    try:
        with open(path, 'r'):
            pass
    except FileNotFoundError:
        raise FileNotFoundError(f"(reader) The file at path '{path}' was not found.")
    except PermissionError:
        raise IOError(f"(reader) Permission denied: cannot access file '{path}'.")
    except IOError as e:
        raise IOError(f"(reader) The file at path '{path}' could not be opened: {e}")

    # Read the file
    reader.SetFileName(path)
    reader.Update()

    # Return the output data
    return reader


def get_point_data_variable(var_name: str, polydata) -> np.ndarray:
    """
    Helper function to get point data variable from polydata.
    
    Parameters
    ----------
    var_name : str
        The name of the variable to extract from the point data.
    polydata : vtk.vtkPolyData
        The polydata object from which to extract the variable. 
    
    Returns
    -------
    np.ndarray
        The variable data as a NumPy array.
        
    Raises
    ------
    ValueError
        If the variable name is not found in the point data.
    """
    try:
        array = polydata.GetPointData().GetArray(var_name)
        if array is None:
            raise ValueError("Array is None")
        return vtk_to_numpy(array)
        
    except:
        # Get available variable names
        point_data = polydata.GetPointData()
        available_vars = [point_data.GetArrayName(i) for i in range(point_data.GetNumberOfArrays())]
        
        raise ValueError(
            f"Variable '{var_name}' not found in point data.\n"
            f"Available variables: {available_vars}"
        )


def read_test(InputConnection, Global_ID_string: str):
    """
    Test function to read global ID and position data from VTK input.
    
    Parameters
    ----------
    InputConnection : vtk.vtkAlgorithmOutput
        The VTK reader output connection.
    Global_ID_string : str
        Name of the global ID variable in the data.
        
    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Global IDs (sorted) and corresponding positions.
        
    Raises
    ------
    ValueError
        If Global_ID_string is None or not found in the data.
    """
    poly_output = InputConnection.GetOutput()

    if Global_ID_string is None:
        raise ValueError("Global_ID_string is None. Please provide a valid string.")
    
    # Get and sort the data by Global ID
    Global_ID = get_point_data_variable(Global_ID_string, poly_output)
    sorted_idx = np.argsort(Global_ID)
    Global_ID_sorted = Global_ID[sorted_idx].astype(np.int32)

    Position = vtk_to_numpy(poly_output.GetPoints().GetData())
    Position_sorted = Position[sorted_idx]
    
    return Global_ID_sorted, Position_sorted


In [92]:
PD = reader("vtp", "/exports/csce/datastore/geos/users/s1857688/Coarse_Graining/pysammos/examples/bedlad_transport/VTU/DES_FB1_0150.vtp") 

FileNotFoundError: (reader) The file at path '/exports/csce/datastore/geos/users/s1857688/Coarse_Graining/pysammos/examples/bedlad_transport/VTU/DES_FB1_0150.vtp' was not found.

In [81]:
def get_point_data_variable(var_name, polydata):
    """
    Helper function to get point data variable from polydata.
    
    Parameters
    ----------
    var_name : str
        The name of the variable to extract from the point data.
    polydata : vtk.vtkPolyData
        The polydata object from which to extract the variable. 
    
    Returns
    -------
    np.ndarray
        The variable data as a NumPy array.
    """
    try:
        array = polydata.GetPointData().GetArray(var_name)
        return vtk_to_numpy(array)
        
    except:
        # Get available variable names
        point_data = polydata.GetPointData()
        available_vars = [point_data.GetArrayName(i) for i in range(point_data.GetNumberOfArrays())]
        
        raise ValueError("get_point_data_variable: Unable to read point data variable.\n"
            f"Variable '{var_name}' not found in point data.\n"
            f"Available variables: {available_vars}"
        )

   
    

In [76]:
def read_test(InputConnection, Global_ID_string):

    poly_output = InputConnection.GetOutput()

    if Global_ID_string is not None:
        # raise an error if Global_ID_string is not found in the file
        
        Global_ID = get_point_data_variable(Global_ID_string, poly_output)
        sorted_idx = np.argsort(Global_ID)
        Global_ID_sorted = Global_ID[sorted_idx].astype(np.int32)

        Position = vtk_to_numpy(poly_output.GetPoints().GetData())
        Position_sorted = Position[sorted_idx]
    else:
        raise ValueError("Global_ID_string is None. Please provide a valid string.")
    
    return Global_ID_sorted, Position_sorted
    

In [83]:
global_id, position = read_test(PD, "Particle_ID")